<a href="https://colab.research.google.com/github/sebastianatanasovici-wq/-notebook-/blob/main/pregunta_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.8 MB/s eta 0:00:00


# **5) Encuesta telefónica**

***Restricciones activas:***

Requerimientos de personas

Límite de llamadas nocturnas

***Interpretación:***

Las llamadas diurnas son preferidas

Las nocturnas se usan solo si ayudan a cumplir restricciones

***Sensibilidad:***

Si baja el costo de llamadas nocturnas → aumentan

Si sube la exigencia de algún grupo → aumenta el total de llamadas

Si se elimina el límite nocturno → cambia la combinación óptima

***Costos***

Llamada día: 2

Llamada noche: 5

***Variables***

x: llamadas en el día

y: llamadas en la noche

In [3]:
def create_ampl_instance(solver: str = 'highs'):
    return ampl_notebook(modules=[solver], license_uuid='default')


def extract_solution(ampl):
    return {
        "Llamadas_dia": round(ampl.get_variable("x").value(), 2),
        "Llamadas_noche": round(ampl.get_variable("y").value(), 2),
        "Costo": round(ampl.get_objective("Costo").value(), 2),
    }


@timed
def solve_encuesta_problem():
    ampl = create_ampl_instance()

    ampl.eval(r'''
        var x >= 0;
        var y >= 0;

        minimize Costo:
            2*x + 5*y;

        r1: 0.3*x + 0.3*y >= 150;
        r2: 0.1*x + 0.3*y >= 120;
        r3: 0.1*x + 0.15*y >= 100;
        r4: 0.1*x + 0.2*y >= 110;

        # máximo mitad en la noche
        r5: y <= 0.5*(x + y);
    ''')

    ampl.option["solver"] = "highs"
    ampl.solve()

    return extract_solution(ampl)


resultado, tiempo = solve_encuesta_problem()

print("=== RESULTADOS EJERCICIO 5 ===")
print(f"Solución -> {resultado}")
print(f"Tiempo   -> {tiempo:.6f} segundos")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 2300
4 simplex iterations
0 barrier iterations
=== RESULTADOS EJERCICIO 5 ===
Solución -> {'Llamadas_dia': 900.0, 'Llamadas_noche': 100.0, 'Costo': 2300.0}
Tiempo   -> 7.450400 segundos
